In [ ]:
%%bash
nproc; free -g | head -2
df -h /tmp /kaggle/working

In [ ]:
%%bash
set -e
LOG=/tmp/pg_build.log; : > $LOG
run() { "$@" >> $LOG 2>&1 || { echo "LỖI khi chạy: $*"; tail -30 $LOG; exit 1; }; }

# Mã nguồn của tác giả: Bao (server + extension), script nạp IMDB, các split
[ -d /tmp/lqo_ml_perspective ] || git clone -q https://github.com/edualc/lqo_ml_perspective.git /tmp/lqo_ml_perspective

PKG=""
[ -x /opt/pg12/bin/pg_config ] && PKG=$(/opt/pg12/bin/pg_config --pkglibdir)
if [ -n "$PKG" ] && [ -f "$PKG/pg_bao.so" ] && [ -f "$PKG/pg_prewarm.so" ]; then
  echo "Đã build từ trước, bỏ qua."
else
  run apt-get update -qq
  run apt-get install -y -qq build-essential libreadline-dev zlib1g-dev
  cd /tmp
  rm -rf postgresql-12.5 postgresql-12.5.tar.gz
  run wget -q https://ftp.postgresql.org/pub/source/v12.5/postgresql-12.5.tar.gz
  tar xzf postgresql-12.5.tar.gz && cd postgresql-12.5
  run ./configure --prefix=/opt/pg12
  run make -j"$(nproc)"
  run make install
  run make -C contrib/pg_prewarm install          # để làm nóng cache
  # Extension của Bao (giống infra/Dockerfile.pg trên Mac)
  rm -rf /tmp/pg_bao && cp -r /tmp/lqo_ml_perspective/bao/pg_extension /tmp/pg_bao
  run make -C /tmp/pg_bao USE_PGXS=1 PG_CONFIG=/opt/pg12/bin/pg_config install
fi
/opt/pg12/bin/postgres --version
PKG=$(/opt/pg12/bin/pg_config --pkglibdir)
[ -f "$PKG/pg_bao.so" ] || { echo "Không thấy pg_bao.so trong $PKG"; exit 1; }
echo "pg_bao OK"

In [ ]:
%%bash
set -e
id -u postgres > /dev/null 2>&1 || useradd -m -s /bin/bash postgres
if [ ! -f /tmp/pgdata/PG_VERSION ]; then
mkdir -p /tmp/pgdata && chown postgres /tmp/pgdata
su postgres -c "/opt/pg12/bin/initdb -D /tmp/pgdata -U postgres --auth=trust" > /dev/null
# Bộ nhớ GIỐNG notebook HybridQO để so hai phương pháp trên cùng môi trường.
# Khác HybridQO: geqo = on (chú thích ¹ Bảng 2: GEQO bật cho Bao và PostgreSQL), nạp pg_bao.
cat >> /tmp/pgdata/postgresql.conf <<'EOF'
listen_addresses = 'localhost'
shared_buffers = 8GB
temp_buffers = 32GB
work_mem = 512MB
effective_cache_size = 32GB
max_parallel_workers_per_gather = 4
max_parallel_maintenance_workers = 4
geqo = on
autovacuum = off
shared_preload_libraries = 'pg_bao'
dynamic_shared_memory_type = mmap
EOF
fi
su postgres -c "/opt/pg12/bin/pg_ctl -D /tmp/pgdata status" > /dev/null \
  || su postgres -c "/opt/pg12/bin/pg_ctl -D /tmp/pgdata -l /tmp/pg.log start -w"
/opt/pg12/bin/psql -U postgres -h localhost -Atc "SELECT name, setting FROM pg_settings WHERE name IN ('shared_buffers','work_mem','geqo','shared_preload_libraries')"

In [ ]:
%%bash
set -e
export PATH=/opt/pg12/bin:$PATH PGHOST=localhost PGUSER=postgres
[ -x /opt/pg12/bin/psql ] || { echo "Chưa có PostgreSQL (phiên Kaggle mới?): chạy lại ô 1 và 2."; exit 1; }
pg_isready -q || { echo "PostgreSQL chưa chạy: chạy lại ô 2."; exit 1; }
if pgrep -f load_job_postgres.sh > /dev/null; then
  echo "Lần chạy trước vẫn đang nạp. Xem: tail /tmp/load.log. Đợi xong rồi chạy lại ô này."; exit 1
fi
if [ -f /tmp/imdb_loaded ]; then
  echo "IMDB đã nạp từ trước, bỏ qua."
else
  dropdb --if-exists imdbload
  rm -rf /tmp/imdb && mkdir -p /tmp/imdb
  wget -q -c -O /tmp/imdb.tgz https://event.cwi.nl/da/job/imdb.tgz
  tar xzf /tmp/imdb.tgz -C /tmp/imdb
  pip install -q absl-py
  python /tmp/lqo_ml_perspective/data/prepend_imdb_headers.py --csv_dir /tmp/imdb 2> /dev/null
  chmod -R a+rX /tmp/imdb /tmp/lqo_ml_perspective
  echo "Đang nạp, khoảng 30–45 phút; log ở /tmp/load.log ..."
  su postgres -c "PATH=/opt/pg12/bin:\$PATH PGHOST=localhost bash /tmp/lqo_ml_perspective/balsa/load-postgres/load_job_postgres.sh /tmp/imdb" > /tmp/load.log 2>&1 &
  PID=$!
  while kill -0 $PID 2> /dev/null; do
    sleep 60
    N=$(grep -c '^COPY ' /tmp/load.log || true)
    if grep -q 'ANALYZE VERBOSE' /tmp/load.log; then STEP="ANALYZE (bước cuối)"
    elif grep -q add_fks /tmp/load.log; then STEP="tạo khóa ngoại"
    else STEP="nạp CSV"; fi
    echo "$(date +%H:%M)  đã nạp $N/21 bảng, đang: $STEP"
  done
  wait $PID || { echo "Script nạp dừng vì lỗi:"; tail -20 /tmp/load.log; exit 1; }
  if grep -q ERROR /tmp/load.log; then echo "Có lỗi khi nạp:"; grep ERROR /tmp/load.log | head -10; exit 1; fi
fi
BAD=""
while read -r t n; do
  c=$(psql -d imdbload -At -v ON_ERROR_STOP=1 -c "SELECT count(*) FROM $t" < /dev/null) \
    || { echo "Không đếm được bảng $t"; exit 1; }
  [ "$c" = "$n" ] || BAD+="$t: có $c, cần $n"$'\n'
done <<'EOF'
aka_name 901343
aka_title 361472
cast_info 36244344
char_name 3140339
comp_cast_type 4
company_name 234997
company_type 4
complete_cast 135086
info_type 113
keyword 134170
kind_type 7
link_type 18
movie_companies 2609129
movie_info 14835720
movie_info_idx 1380035
movie_keyword 4523930
movie_link 29997
name 4167491
person_info 2963664
role_type 12
title 2528312
EOF
[ -z "$BAD" ] || { echo "Các bảng sau sai số dòng:"; echo "$BAD"; exit 1; }
echo "Đủ 21 bảng, số dòng khớp."
touch /tmp/imdb_loaded
rm -rf /tmp/imdb /tmp/imdb.tgz

In [ ]:
%%bash
set -e
pip install -q uv
[ -x /tmp/bao_env/bin/python ] || uv venv -q -p 3.10 /tmp/bao_env
# Giống lqo_ml_perspective/bao/requirements.txt (torch bản CPU)
uv pip install -q --python /tmp/bao_env/bin/python torch==1.13.1 --index-url https://download.pytorch.org/whl/cpu
uv pip install -q --python /tmp/bao_env/bin/python numpy==1.23.5 scikit-learn==1.1.3 scipy==1.9.3 joblib==1.2.0 \
  threadpoolctl==3.1.0 psycopg2-binary==2.9.5
/tmp/bao_env/bin/python -c "import torch, sklearn, psycopg2; print('torch', torch.__version__, '| sklearn', sklearn.__version__)"

In [ ]:
%%writefile /tmp/ablation_interleaved.py
"""Ablation 8.4 (tắt bitmap + tid scan) và 8.5 (tắt GEQO), đo XEN KẼ để loại bỏ trôi môi trường.

Với mỗi câu JOB, chạy liền nhau cả 3 cấu hình (thứ tự xoay vòng theo câu để không cấu hình nào luôn chạy trước):
  default            : cấu hình mặc định (GEQO bật, bitmap/tid scan bật)
  no_geqo            : SET geqo TO off
  no_bitmap_tidscan  : SET enable_bitmapscan TO off; SET enable_tidscan TO off
Mỗi cấu hình: kết nối mới, SET theo phiên (không đổi cấu hình CSDL), EXPLAIN ANALYZE 3 lần, timeout 180 s,
giống run_test_queries.py của tác giả. Ghi ra 3 file txt cùng định dạng log của tác giả để dùng lại c1_win_loss.py.

Chạy TRONG container bao_server:
  python3 /app/scripts/ablation_interleaved.py --tag interleaved_YYYYMMDD
"""
import argparse
import datetime
import glob
import os
import sys

import psycopg2

TIMEOUT_MS = 3 * 60 * 1000
RUNS = 3
CONFIGS = {
    'default': [],
    'no_geqo': ["SET geqo TO off"],
    'no_bitmap_tidscan': ["SET enable_bitmapscan TO off", "SET enable_tidscan TO off"],
}
DSN = "dbname={db} user=postgres password=postgres host=localhost"


def other_connections(db):
    conn = psycopg2.connect(DSN.format(db=db))
    cur = conn.cursor()
    cur.execute("SELECT count(*) FROM pg_stat_activity "
                "WHERE backend_type = 'client backend' AND pid <> pg_backend_pid()")
    n = cur.fetchone()[0]
    conn.close()
    return n


def run(sql, db, sets):
    out = []
    conn = psycopg2.connect(DSN.format(db=db))
    try:
        cur = conn.cursor()
        cur.execute(f"SET statement_timeout TO {TIMEOUT_MS}")
        cur.execute("SET enable_bao TO off")
        for s in sets:
            cur.execute(s)
        for _ in range(RUNS):
            cur.execute(f"EXPLAIN (ANALYZE, VERBOSE, FORMAT JSON) {sql}")
            r = cur.fetchall()[0][0][-1]
            out.append((r['Planning Time'], r['Execution Time']))
    except Exception as e:  # timeout: ghi như tác giả (2 × timeout)
        print("  timeout/lỗi:", str(e).splitlines()[0], flush=True)
        out = [(2 * TIMEOUT_MS, 2 * TIMEOUT_MS)] * RUNS
    finally:
        conn.close()
    return out


def main(a):
    n = other_connections(a.db)
    if n and not a.force:
        sys.exit(f"Đang có {n} kết nối khác vào {a.db}. Dừng chúng trước (hoặc --force).")
    paths = sorted(glob.glob(os.path.join(a.query_dir, '*.sql')))
    print(f"{len(paths)} câu, cấu hình: {list(CONFIGS)}", flush=True)
    files = {c: open(os.path.join(a.out_dir, f"test__postgres__job__{c}__{a.tag}.txt"), 'w') for c in CONFIGS}
    names = list(CONFIGS)
    for qi, p in enumerate(paths):
        sql = open(p).read()
        rel = 'queries/join-order-benchmark/' + os.path.basename(p)
        order = names[qi % 3:] + names[:qi % 3]
        line = []
        for c in order:
            ms = run(sql, a.db, CONFIGS[c])
            ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')
            for i, (pt, et) in enumerate(ms):
                files[c].write(f"x, {i}, {ts}, {rel}, {pt}, {et}, PG\n")
            files[c].flush()
            line.append(f"{c}={ms[-1][0] + ms[-1][1]:.0f}ms")
        print(f"[{qi + 1}/{len(paths)}] {os.path.basename(p)}  " + "  ".join(line), flush=True)
    for f in files.values():
        f.close()
    print("Xong:", [f.name for f in files.values()])


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--db', default='imdbload')
    ap.add_argument('--query_dir', default='/app/bao/queries/join-order-benchmark')
    ap.add_argument('--out_dir', default='/app/logs')
    ap.add_argument('--tag', required=True)
    ap.add_argument('--force', action='store_true')
    main(ap.parse_args())


In [ ]:
%%bash
set -e
export PATH=/opt/pg12/bin:$PATH PGHOST=localhost PGUSER=postgres
psql -d imdbload -qc "CREATE EXTENSION IF NOT EXISTS pg_prewarm"
psql -d imdbload -qAt -c "SELECT count(pg_prewarm(c.oid,'read')) FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace WHERE n.nspname='public' AND c.relkind IN ('r','i')" > /dev/null
T0=$(date +%s)
for f in /tmp/lqo_ml_perspective/bao/queries/join-order-benchmark/*.sql; do
  psql -d imdbload -q -o /dev/null -c "SET statement_timeout = 300000;" -f "$f" || echo "  $(basename $f): lỗi/timeout"
done
echo "Đã làm nóng: chạy mỗi câu JOB một lần trong $(( $(date +%s) - T0 )) giây"

In [ ]:
%%bash
# Ablation 8.4 + 8.5: mỗi câu chạy liền 3 cấu hình (default / no_geqo / no_bitmap_tidscan), thứ tự xoay vòng, 3 lần mỗi
# cấu hình, timeout 180 s — giống hệt lần chạy trên Mac (scripts/ablation_interleaved.py).
mkdir -p /kaggle/working/ablation_logs
/tmp/bao_env/bin/python -u /tmp/ablation_interleaved.py --tag kaggle \
  --query_dir /tmp/lqo_ml_perspective/bao/queries/join-order-benchmark --out_dir /kaggle/working/ablation_logs

In [ ]:
import pandas as pd, os
D = "/kaggle/working/ablation_logs"
def load(c):
    rows = [l.split(",") for l in open(f"{D}/test__postgres__job__{c}__kaggle.txt")]
    d = pd.DataFrame({"q": [os.path.basename(r[3].strip())[:-4] for r in rows], "run": [int(r[1]) for r in rows],
                      "ms": [float(r[4]) + float(r[5]) for r in rows]})
    return d[d.run == 2].set_index("q").ms
base = load("default")
for c in ["no_geqo", "no_bitmap_tidscan"]:
    x = load(c); r = (x / base).sort_values()
    print(f"{c}: tổng {base.sum()/1000:.1f} s → {x.sum()/1000:.1f} s; chậm đi >2×: {(r > 2).sum()} câu "
          f"({', '.join(r[r > 2].index[-8:])}); nhanh lên >2×: {(r < 0.5).sum()} câu")